In [1]:
import pandas as pd
import requests
from io import BytesIO

In [3]:
from data_pipelines.dags.medication_etl_src.api.api_anvisa import ApiAnvisa
from data_pipelines.dags.medication_etl_src.api.adapter.anvisa.anvisa_medicines_adapter import AnvisaMedicinesAdapter
from data_pipelines.dags.medication_etl_src.api.api_anvisa import ApiAnvi

ModuleNotFoundError: No module named 'medication_etl_src'

In [ ]:
file2 = requests.get("https://consultas.anvisa.gov.br/api/consulta/medicamento/download?filter[situacaoRegistro]=V")
outro = pd.read_excel(BytesIO(file2.content))

In [24]:
SISTEMA_CONSULTAS = outro

In [4]:
#DADOS_ABERTOS_MEDICAMENTOS = pd.read_csv('C:/Users/Marcelo/Desktop/Medicamentos/extracao-dados-medicamentos/airflow/dags/csvs/DADOS_ABERTOS_MEDICAMENTOS_05_11_24.csv', encoding="ISO-8859-1", delimiter=";", decimal=",")

In [25]:
#file = requests.get('https://dados.anvisa.gov.br/dados/TA_PRECO_MEDICAMENTO_GOV.csv', verify=False)
#__ta_precos = pd.read_csv(BytesIO(file.content), header=52, delimiter=";", decimal=",", low_memory=False)
#ta_precos = pd.read_csv('C:/Users/Marcelo/Desktop/Medicamentos/extracao-dados-medicamentos/airflow/dags/csvs/TA_PRECO_MEDICAMENTO.csv', header=41, delimiter=";", decimal=",", low_memory=False)

In [33]:
res = requests.get("https://consultas.anvisa.gov.br/api/consulta/medicamento/produtos?count=100&filter[situacaoRegistro]=V&filter[tipoAutorizacao]=REGISTRADO&page=1", headers={"Authorization": "Guest"})

In [34]:
df_t = pd.DataFrame(res.json()['content'])

In [21]:
len(df)

10472

In [ ]:
df

In [3]:
api = ApiAnvisa()

In [4]:
result = api.get_medicines()

In [5]:
len(result)

10472

In [6]:
df = pd.DataFrame(result)

In [18]:
df

,ordem,produto,empresa,processo
0,1,"{'codigo': 2762872, 'nome': '18F-PSMA1007', 'n...","{'cnpj': '08820007000161', 'razaoSocial': 'CMR...","{'numero': '25351093523202175', 'situacao': 90..."
1,2,"{'codigo': 765613, 'nome': 'A SAÚDE DA MULHER'...","{'cnpj': '57507378000365', 'razaoSocial': 'EMS...","{'numero': '25351668917201032', 'situacao': 29..."
2,3,"{'codigo': 3548748, 'nome': 'AAS', 'numeroRegi...","{'cnpj': '61082426000207', 'razaoSocial': 'COS...","{'numero': '25351497156202266', 'situacao': 29..."
3,4,"{'codigo': 3529192, 'nome': 'AAS PROTECT', 'nu...","{'cnpj': '61082426000207', 'razaoSocial': 'COS...","{'numero': '25351497237202266', 'situacao': 29..."
4,5,"{'codigo': 428320, 'nome': 'ABC', 'numeroRegis...","{'cnpj': '92695691000103', 'razaoSocial': 'KLE...","{'numero': '25351141033200572', 'situacao': 29..."
...,...,...,...,...
10467,10468,"{'codigo': 666926, 'nome': 'ácido zoledrônico'...","{'cnpj': '61190096000192', 'razaoSocial': 'EUR...","{'numero': '25351515342200983', 'situacao': 29..."
10468,10469,"{'codigo': 903696, 'nome': 'ácido zoledrônico'...","{'cnpj': '49324221000104', 'razaoSocial': 'FRE...","{'numero': '25351389723201352', 'situacao': 29..."
10469,10470,"{'codigo': 796006, 'nome': 'ácido zoledrônico'...","{'cnpj': '05035244000123', 'razaoSocial': 'SUN...","{'numero': '25351686804201172', 'situacao': 29..."
10470,10471,"{'codigo': 1012584, 'nome': 'ácido zoledrônico...","{'cnpj': '11643096000122', 'razaoSocial': 'VIA...","{'numero': '25351771065201417', 'situacao': 29..."


In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10472 entries, 0 to 10471
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   ordem     10472 non-null  int64 
 1   produto   10472 non-null  object
 2   empresa   10472 non-null  object
 3   processo  10472 non-null  object
dtypes: int64(1), object(3)
memory usage: 327.4+ KB


In [23]:
df = pd.concat([
    df.drop(columns=['produto', 'empresa', 'processo']),
    pd.json_normalize(df['produto']),
    pd.json_normalize(df['empresa']),
    pd.json_normalize(df['processo']),
], axis=1)

In [38]:
df = df[df["tipoAutorizacao"] != "NOTIFICADO"]

In [40]:
df.iloc[0]

ordem                                                                             2
codigo                                                                       765613
nome                                                              A SAÚDE DA MULHER
numeroRegistro                                                            102351059
categoria                                                                      None
situacaoRotulo                                                                 None
dataVencimento                                                                 None
mesAnoVencimento                                                             072029
dataVencimentoRegistro                                 2029-07-01T00:00:00.000-0300
principioAtivo                    TINTURA AGONIADA PLUMÉRIA, SALICILATO DE SÓDIO...
situacaoApresentacao                                                          Ativo
dataRegistro                                           2012-03-26T00:00:00.0

In [13]:
df

,ordem,produto,empresa,processo
0,1,"{'codigo': 2762872, 'nome': '18F-PSMA1007', 'n...","{'cnpj': '08820007000161', 'razaoSocial': 'CMR...","{'numero': '25351093523202175', 'situacao': 90..."
1,2,"{'codigo': 765613, 'nome': 'A SAÚDE DA MULHER'...","{'cnpj': '57507378000365', 'razaoSocial': 'EMS...","{'numero': '25351668917201032', 'situacao': 29..."
2,3,"{'codigo': 3548748, 'nome': 'AAS', 'numeroRegi...","{'cnpj': '61082426000207', 'razaoSocial': 'COS...","{'numero': '25351497156202266', 'situacao': 29..."
3,4,"{'codigo': 3529192, 'nome': 'AAS PROTECT', 'nu...","{'cnpj': '61082426000207', 'razaoSocial': 'COS...","{'numero': '25351497237202266', 'situacao': 29..."
4,5,"{'codigo': 428320, 'nome': 'ABC', 'numeroRegis...","{'cnpj': '92695691000103', 'razaoSocial': 'KLE...","{'numero': '25351141033200572', 'situacao': 29..."
...,...,...,...,...
10467,10468,"{'codigo': 666926, 'nome': 'ácido zoledrônico'...","{'cnpj': '61190096000192', 'razaoSocial': 'EUR...","{'numero': '25351515342200983', 'situacao': 29..."
10468,10469,"{'codigo': 903696, 'nome': 'ácido zoledrônico'...","{'cnpj': '49324221000104', 'razaoSocial': 'FRE...","{'numero': '25351389723201352', 'situacao': 29..."
10469,10470,"{'codigo': 796006, 'nome': 'ácido zoledrônico'...","{'cnpj': '05035244000123', 'razaoSocial': 'SUN...","{'numero': '25351686804201172', 'situacao': 29..."
10470,10471,"{'codigo': 1012584, 'nome': 'ácido zoledrônico...","{'cnpj': '11643096000122', 'razaoSocial': 'VIA...","{'numero': '25351771065201417', 'situacao': 29..."


In [7]:
df2 = AnvisaMedicinesAdapter().adapt(df)

In [8]:
df2

,codigo_anvisa,nome_produto,numero_registro,numero_processo,principio_ativo,categoria_regulatoria,cnpj_laboratorio,razao_social_laboratorio
0,2762872,18F-PSMA1007,None,25351093523202175,AMESPRO FLUORNICOTINAMIDA (18 F),Radiofármaco,08820007000161,CMR CAMPINAS PHARMA LTDA
1,765613,A SAÚDE DA MULHER,102351059,25351668917201032,"TINTURA AGONIADA PLUMÉRIA, SALICILATO DE SÓDIO...",Novo,57507378000365,EMS S/A
2,3548748,AAS,178170936,25351497156202266,ACIDO ACETILSALICILICO,Similar,61082426000207,COSMED INDUSTRIA DE COSMETICOS E MEDICAMENTOS ...
3,3529192,AAS PROTECT,178170931,25351497237202266,ÁCIDO ACETILSALICÍLICO,Similar,61082426000207,COSMED INDUSTRIA DE COSMETICOS E MEDICAMENTOS ...
4,428320,ABC,106890153,25351141033200572,CLOTRIMAZOL,Similar,92695691000103,KLEY HERTZ FARMACEUTICA S.A
...,...,...,...,...,...,...,...,...
10467,666926,ácido zoledrônico,100431026,25351515342200983,ácido zoledrônico monoidratado,Genérico,61190096000192,EUROFARMA LABORATÓRIOS S.A.
10468,903696,ácido zoledrônico,100410162,25351389723201352,ácido zoledrônico monoidratado,Genérico,49324221000104,FRESENIUS KABI BRASIL LTDA
10469,796006,ácido zoledrônico,146820032,25351686804201172,ácido zoledrônico monoidratado,Genérico,05035244000123,SUN FARMACÊUTICA DO BRASIL LTDA
10470,1012584,ácido zoledrônico,188300060,25351771065201417,ácido zoledrônico monoidratado,Genérico,11643096000122,VIATRIS FARMACEUTICA DO BRASIL LTDA


In [11]:
%%timeit
1+1

11.4 ns ± 0.663 ns per loop (mean ± std. dev. of 7 runs, 100,000,000 loops each)


In [18]:
from dataclasses import dataclass, asdict

@dataclass
class Medicine:

    codigo_anvisa: str = None
    nome_produto: str = None
    numero_registro: str = None
    numero_processo: str = None
    principio_ativo: str = None
    categoria_regulatoria: str = None
    cnpj_laboratorio: str = None
    razao_social_laboratorio: str = None



In [29]:
from dataclasses import dataclass, asdict

In [24]:
%%timeit
medicines = df2.apply(lambda row: Medicine(*row), axis=1)

84.3 ms ± 3.64 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [26]:
medicines = df2.apply(lambda row: Medicine(*row), axis=1).tolist()

In [28]:
medicines[0]

Medicine(codigo_anvisa=2762872, nome_produto='18F-PSMA1007', numero_registro=None, numero_processo='25351093523202175', principio_ativo='AMESPRO FLUORNICOTINAMIDA (18 F)', categoria_regulatoria='Radiofármaco', cnpj_laboratorio='08820007000161', razao_social_laboratorio='CMR CAMPINAS PHARMA LTDA')

In [32]:
df_ = pd.DataFrame.from_records([asdict(s) for s in medicines])

In [35]:
df_.to_csv("teste123.csv", index=False)

In [36]:
pd.read_csv("teste123.csv")

,codigo_anvisa,nome_produto,numero_registro,numero_processo,principio_ativo,categoria_regulatoria,cnpj_laboratorio,razao_social_laboratorio
0,2762872,18F-PSMA1007,NaN,25351093523202175,AMESPRO FLUORNICOTINAMIDA (18 F),Radiofármaco,8820007000161,CMR CAMPINAS PHARMA LTDA
1,765613,A SAÚDE DA MULHER,102351059.0,25351668917201032,"TINTURA AGONIADA PLUMÉRIA, SALICILATO DE SÓDIO...",Novo,57507378000365,EMS S/A
2,3548748,AAS,178170936.0,25351497156202266,ACIDO ACETILSALICILICO,Similar,61082426000207,COSMED INDUSTRIA DE COSMETICOS E MEDICAMENTOS ...
3,3529192,AAS PROTECT,178170931.0,25351497237202266,ÁCIDO ACETILSALICÍLICO,Similar,61082426000207,COSMED INDUSTRIA DE COSMETICOS E MEDICAMENTOS ...
4,428320,ABC,106890153.0,25351141033200572,CLOTRIMAZOL,Similar,92695691000103,KLEY HERTZ FARMACEUTICA S.A
...,...,...,...,...,...,...,...,...
10467,666926,ácido zoledrônico,100431026.0,25351515342200983,ácido zoledrônico monoidratado,Genérico,61190096000192,EUROFARMA LABORATÓRIOS S.A.
10468,903696,ácido zoledrônico,100410162.0,25351389723201352,ácido zoledrônico monoidratado,Genérico,49324221000104,FRESENIUS KABI BRASIL LTDA
10469,796006,ácido zoledrônico,146820032.0,25351686804201172,ácido zoledrônico monoidratado,Genérico,5035244000123,SUN FARMACÊUTICA DO BRASIL LTDA
10470,1012584,ácido zoledrônico,188300060.0,25351771065201417,ácido zoledrônico monoidratado,Genérico,11643096000122,VIATRIS FARMACEUTICA DO BRASIL LTDA
